# Dataset Audit and Visualisation: Class Distribution, Image Quality and Annotation Validation

**Dataset:** COCO128 (128 real-world images with YOLO-format bounding-box annotations)  
**Task focus:** Audit vehicle-related object classes — bicycle, car, motorcycle, bus, train and truck.

This notebook downloads the dataset automatically, checks its structure, calculates audit metrics from the actual files, and creates graphs. Results are **computed when you run the notebook**; no sample counts or graph values are hard-coded.

> COCO128 is a small subset of COCO commonly used for object-detection experiments. It contains multiple object categories, so this notebook focuses the class-distribution analysis on vehicle categories while still validating all annotations.


## 1. Install and import libraries

In [ ]:
!pip -q install opencv-python-headless pyyaml seaborn

In [ ]:
import os
import zipfile
import urllib.request
from pathlib import Path
from collections import Counter

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

## 2. Download and extract the real dataset

The archive is downloaded from the Ultralytics COCO128 dataset asset. If it is already present in the Colab session, the download is skipped.

In [ ]:
DATA_URL = "https://ultralytics.com/assets/coco128.zip"
ZIP_PATH = Path("/content/coco128.zip")
EXTRACT_DIR = Path("/content/datasets")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    print("Downloading COCO128...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
else:
    print("Dataset archive already exists.")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

DATASET_DIR = EXTRACT_DIR / "coco128"
IMAGE_DIR = DATASET_DIR / "images" / "train2017"
LABEL_DIR = DATASET_DIR / "labels" / "train2017"
YAML_PATH = DATASET_DIR / "coco128.yaml"

print("Dataset extracted.")
print("Image folder exists:", IMAGE_DIR.exists())
print("Label folder exists:", LABEL_DIR.exists())
print("YAML file exists:", YAML_PATH.exists())

## 3. Dataset overview and class names

In [ ]:
if not IMAGE_DIR.exists() or not LABEL_DIR.exists():
    raise FileNotFoundError("Expected COCO128 images/labels folders were not found. Check the downloaded archive structure.")

with open(YAML_PATH, "r") as f:
    data_config = yaml.safe_load(f)

names = data_config["names"]
if isinstance(names, list):
    class_names = {i: name for i, name in enumerate(names)}
else:
    class_names = {int(k): v for k, v in names.items()}

image_files = sorted([p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
label_files = sorted(LABEL_DIR.glob("*.txt"))

print(f"Number of images found: {len(image_files)}")
print(f"Number of label files found: {len(label_files)}")
print(f"Number of classes defined: {len(class_names)}")
print("\nVehicle classes included in the audit:")
vehicle_names = {"bicycle", "car", "motorcycle", "bus", "train", "truck"}
vehicle_ids = {i: n for i, n in class_names.items() if n in vehicle_names}
print(vehicle_ids)

## 4. Class distribution audit

YOLO label rows use the format:

`class_id x_center y_center width height`

The coordinates are expected to be normalized to the range 0–1. The graph below counts annotated objects, not images. It displays vehicle classes only.

In [ ]:
all_class_counts = Counter()
vehicle_counts = Counter()

for label_path in label_files:
    try:
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
    except Exception:
        continue
    for line in lines:
        parts = line.split()
        if len(parts) == 5:
            try:
                class_id = int(float(parts[0]))
                all_class_counts[class_id] += 1
                if class_id in vehicle_ids:
                    vehicle_counts[class_id] += 1
            except ValueError:
                pass

class_distribution = pd.DataFrame([
    {"Class": class_names.get(cid, f"Unknown ({cid})"), "Class ID": cid, "Object Count": vehicle_counts.get(cid, 0)}
    for cid in vehicle_ids
]).sort_values("Object Count", ascending=False)

display(class_distribution)

plt.figure(figsize=(10, 5))
sns.barplot(data=class_distribution, x="Class", y="Object Count")
plt.title("Vehicle Class Distribution (Annotated Objects)")
plt.xlabel("Vehicle class")
plt.ylabel("Number of annotated objects")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

print("Total vehicle annotations:", int(class_distribution["Object Count"].sum()))

## 5. Image quality audit

For every readable image, the notebook measures:
- **Width and height** in pixels
- **Brightness:** mean grayscale intensity (0–255)
- **Blur indicator:** variance of the Laplacian; lower values can indicate a blurrier image

These are simple screening indicators, not absolute image-quality grades. A low Laplacian variance does not always mean an image is unusable.

In [ ]:
image_metrics = []
unreadable_images = []

for image_path in image_files:
    image = cv2.imread(str(image_path))
    if image is None:
        unreadable_images.append(image_path.name)
        continue

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape
    brightness = float(np.mean(gray))
    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    image_metrics.append({
        "Image": image_path.name,
        "Width": width,
        "Height": height,
        "Brightness": brightness,
        "Blur Score": blur_score,
        "Resolution": width * height
    })

image_quality_df = pd.DataFrame(image_metrics)
display(image_quality_df.head())

print("Readable images:", len(image_quality_df))
print("Unreadable images:", len(unreadable_images))
if unreadable_images:
    print("Unreadable file names:", unreadable_images[:10])

if not image_quality_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    sns.histplot(image_quality_df["Brightness"], bins=20, kde=True, ax=axes[0])
    axes[0].set_title("Image Brightness Distribution")
    axes[0].set_xlabel("Mean grayscale intensity")

    sns.histplot(image_quality_df["Blur Score"], bins=20, kde=True, ax=axes[1])
    axes[1].set_title("Blur Score Distribution")
    axes[1].set_xlabel("Variance of Laplacian")

    sns.scatterplot(data=image_quality_df, x="Width", y="Height", ax=axes[2])
    axes[2].set_title("Image Dimensions")
    axes[2].set_xlabel("Width (pixels)")
    axes[2].set_ylabel("Height (pixels)")

    plt.tight_layout()
    plt.show()

    print("\nImage quality summary:")
    display(image_quality_df[["Width", "Height", "Brightness", "Blur Score"]].describe().round(2))

## 6. Annotation validation

The validation checks:
1. Each non-empty annotation row has exactly five values.
2. Class ID is an integer and exists in the dataset class list.
3. Bounding-box values are numeric.
4. Normalized coordinates are between 0 and 1.
5. Width and height are greater than zero.
6. Each image has a matching label file, and each label file has a matching image.

An empty label file is treated as a valid image with no annotated objects. A missing label file is reported separately.

In [ ]:
image_stems = {p.stem for p in image_files}
label_stems = {p.stem for p in label_files}

missing_label_for_image = sorted(image_stems - label_stems)
label_without_image = sorted(label_stems - image_stems)

validation_rows = []
for label_path in label_files:
    for line_number, line in enumerate(label_path.read_text(errors="replace").splitlines(), start=1):
        if not line.strip():
            continue

        parts = line.split()
        errors = []

        if len(parts) != 5:
            errors.append("Expected 5 values")
        else:
            try:
                class_value = float(parts[0])
                class_id = int(class_value)
                if class_value != class_id:
                    errors.append("Class ID is not an integer")
                elif class_id not in class_names:
                    errors.append("Unknown class ID")
            except ValueError:
                errors.append("Class ID is not numeric")
                class_id = None

            try:
                box = [float(x) for x in parts[1:]]
                if len(box) == 4:
                    xc, yc, w, h = box
                    if not all(np.isfinite(box)):
                        errors.append("Non-finite coordinate")
                    if not all(0 <= value <= 1 for value in box):
                        errors.append("Coordinate outside [0, 1]")
                    if w <= 0 or h <= 0:
                        errors.append("Non-positive box width/height")
                    if w > 1 or h > 1:
                        errors.append("Box size exceeds image")
            except ValueError:
                errors.append("Non-numeric bounding-box value")

        validation_rows.append({
            "Label File": label_path.name,
            "Line": line_number,
            "Valid": len(errors) == 0,
            "Errors": "; ".join(errors)
        })

validation_df = pd.DataFrame(validation_rows, columns=["Label File", "Line", "Valid", "Errors"])
valid_count = int(validation_df["Valid"].sum()) if not validation_df.empty else 0
invalid_count = int((~validation_df["Valid"]).sum()) if not validation_df.empty else 0

print("Images without label files:", len(missing_label_for_image))
print("Label files without matching images:", len(label_without_image))
print("Annotation rows checked:", len(validation_df))
print("Valid annotation rows:", valid_count)
print("Invalid annotation rows:", invalid_count)

if not validation_df.empty:
    display(validation_df[validation_df["Valid"] == False].head(20))

validation_summary = pd.DataFrame({
    "Status": ["Valid rows", "Invalid rows", "Images missing labels", "Labels without images"],
    "Count": [valid_count, invalid_count, len(missing_label_for_image), len(label_without_image)]
})
display(validation_summary)

plt.figure(figsize=(8, 4))
sns.barplot(data=validation_summary, x="Status", y="Count")
plt.title("Annotation Validation Summary")
plt.ylabel("Count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 7. Visual sample: images with bounding boxes

This section draws YOLO annotations on a few sample images so you can visually inspect whether the labels align with the objects.

In [ ]:
def draw_yolo_boxes(image_path, label_path):
    image = cv2.imread(str(image_path))
    if image is None:
        return None

    h, w = image.shape[:2]
    if label_path.exists():
        for line in label_path.read_text(errors="replace").splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            try:
                cid = int(float(parts[0]))
                xc, yc, bw, bh = map(float, parts[1:])
            except ValueError:
                continue

            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)

            x1, x2 = max(0, x1), min(w - 1, x2)
            y1, y2 = max(0, y1), min(h - 1, y2)

            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            name = class_names.get(cid, str(cid))
            cv2.putText(image, name, (x1, max(15, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

sample_images = image_files[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, image_path in zip(axes.flat, sample_images):
    label_path = LABEL_DIR / f"{image_path.stem}.txt"
    shown = draw_yolo_boxes(image_path, label_path)
    if shown is not None:
        ax.imshow(shown)
        ax.set_title(image_path.name)
    ax.axis("off")

for ax in axes.flat[len(sample_images):]:
    ax.axis("off")

plt.suptitle("Sample Images with YOLO Bounding Boxes")
plt.tight_layout()
plt.show()

## 8. Final audit report

Use the generated tables and plots to describe the dataset's class balance, image characteristics, and annotation quality. The report below is generated from the audit results.

In [ ]:
print("========== DATASET AUDIT REPORT ==========")
print(f"Dataset: COCO128")
print(f"Images discovered: {len(image_files)}")
print(f"Readable images: {len(image_quality_df)}")
print(f"Unreadable images: {len(unreadable_images)}")
print(f"Label files: {len(label_files)}")
print(f"Vehicle object annotations: {int(class_distribution['Object Count'].sum())}")
print(f"Annotation rows checked: {len(validation_df)}")
print(f"Valid annotation rows: {valid_count}")
print(f"Invalid annotation rows: {invalid_count}")
print(f"Images missing labels: {len(missing_label_for_image)}")
print(f"Labels without matching images: {len(label_without_image)}")

if not image_quality_df.empty:
    print(f"Average brightness: {image_quality_df['Brightness'].mean():.2f}")
    print(f"Average blur score: {image_quality_df['Blur Score'].mean():.2f}")

print("\nVehicle class counts:")
display(class_distribution)
print("==========================================")

## 9. Download audit tables

The following cell saves the computed audit tables as CSV files and bundles them into one ZIP file for download from Colab.

In [ ]:
from google.colab import files

class_distribution.to_csv("/content/vehicle_class_distribution.csv", index=False)
image_quality_df.to_csv("/content/image_quality_metrics.csv", index=False)
validation_df.to_csv("/content/annotation_validation.csv", index=False)
validation_summary.to_csv("/content/annotation_validation_summary.csv", index=False)

import zipfile
with zipfile.ZipFile("/content/dataset_audit_results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for filename in [
        "vehicle_class_distribution.csv",
        "image_quality_metrics.csv",
        "annotation_validation.csv",
        "annotation_validation_summary.csv"
    ]:
        zf.write(f"/content/{filename}", arcname=filename)

files.download("/content/dataset_audit_results.zip")